# Benchmark Analysis

Compare your portfolio's risk-adjusted performance against a market benchmark. This notebook calculates alpha, beta, tracking error, information ratio, up/down capture ratio, and correlation, and visualises the relationship through cumulative growth, rolling beta, and excess return scatter plots.

In [ ]:
%pip install yfinance numpy pandas matplotlib -q

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timedelta

In [ ]:
# Create your portfolio here
tickers = ['SPY','QQQ']
weights = [0.60, 0.40]
portfolio = dict(zip(tickers, weights))
print('Portfolio:', portfolio)

# Benchmark ticker
benchmark_ticker = 'SPY'

# Date range
start_date = '2000-01-01'
end_date = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
print('Start Date:', start_date)
print('End Date  :', end_date)

# Risk-free rate
risk_free_rate = 0.02
print('Benchmark:     ', benchmark_ticker)
print('Risk-free Rate:', risk_free_rate)

## Data Loading

In [ ]:
# Download portfolio tickers
data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True, progress=False)
if isinstance(data.columns, pd.MultiIndex):
    data = data['Close']

# Download benchmark
bench_raw = yf.download(benchmark_ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)
if isinstance(bench_raw.columns, pd.MultiIndex):
    bench_raw = bench_raw['Close']
bench_raw = bench_raw.squeeze()

# Align available tickers with weights
available = [t for t in tickers if t in data.columns]
w_aligned = [weights[tickers.index(t)] for t in available]
w_sum = sum(w_aligned)
w_aligned = [w / w_sum for w in w_aligned]
data = data[available]

# Daily returns
returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)
p_ret = pd.Series(returns.dot(w_aligned), name='Portfolio')
b_ret = bench_raw.pct_change(fill_method=None).dropna()

# Align on common dates
common = p_ret.index.intersection(b_ret.index)
p_ret = p_ret.loc[common]
b_ret = b_ret.loc[common]

print(f'Data shape: {len(p_ret)} trading days')
print(f'Portfolio  : {p_ret.index[0].date()} → {p_ret.index[-1].date()}')
print(f'Benchmark  : {b_ret.index[0].date()} → {b_ret.index[-1].date()}')

## Metric Functions

In [ ]:
def ann_return(r):
    return (1 + r).prod() ** (252 / len(r)) - 1

def ann_vol(r):
    return r.std() * np.sqrt(252)

def sharpe(r, rf=risk_free_rate):
    vol = ann_vol(r)
    return (ann_return(r) - rf) / vol if vol > 0 else float('nan')

def max_drawdown(r):
    cum = (1 + r).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min()

def beta(p, b):
    cov = np.cov(p, b)
    return cov[0, 1] / cov[1, 1] if cov[1, 1] != 0 else float('nan')

def alpha_jensen(p, b, rf=risk_free_rate):
    b_val = beta(p, b)
    return ann_return(pd.Series(p)) - (rf + b_val * (ann_return(pd.Series(b)) - rf))

def tracking_error(p, b):
    return (p - b).std() * np.sqrt(252)

def information_ratio(p, b):
    te = tracking_error(p, b)
    return (ann_return(p) - ann_return(b)) / te if te > 0 else float('nan')

def up_capture(p, b):
    mask = b > 0
    return (p[mask].mean() / b[mask].mean() * 100) if mask.sum() > 0 else float('nan')

def down_capture(p, b):
    mask = b < 0
    return (p[mask].mean() / b[mask].mean() * 100) if mask.sum() > 0 else float('nan')

## Benchmark Metrics Table

In [ ]:
corr_coef = float(np.corrcoef(p_ret.values, b_ret.values)[0, 1])
b_val = beta(p_ret.values, b_ret.values)
a_val = alpha_jensen(p_ret.values, b_ret.values)
te    = tracking_error(p_ret, b_ret)
ir    = information_ratio(p_ret, b_ret)
uc    = up_capture(p_ret, b_ret)
dc    = down_capture(p_ret, b_ret)

metrics = [
    ('Annual Return',     f'{ann_return(p_ret):.2%}',       f'{ann_return(b_ret):.2%}'),
    ('Annual Volatility', f'{ann_vol(p_ret):.2%}',          f'{ann_vol(b_ret):.2%}'),
    ('Sharpe Ratio',      f'{sharpe(p_ret):.3f}',           f'{sharpe(b_ret):.3f}'),
    ('Max Drawdown',      f'{max_drawdown(p_ret):.2%}',     f'{max_drawdown(b_ret):.2%}'),
    ('Beta',              f'{b_val:.3f}',                   '1.000'),
    ('Alpha (Jensen)',    f'{a_val:.2%}',                   '0.00%'),
    ('Tracking Error',    f'{te:.2%}',                      '0.00%'),
    ('Information Ratio', f'{ir:.3f}',                      '———'),
    ('Up Capture',        f'{uc:.1f}%',                     '100.0%'),
    ('Down Capture',      f'{dc:.1f}%',                     '100.0%'),
    ('Correlation',       f'{corr_coef:.3f}',               '1.000'),
    ('R-Squared',         f'{corr_coef**2:.3f}',            '1.000'),
]
metrics_df = pd.DataFrame(metrics, columns=['Metric', 'Portfolio', f'Benchmark ({benchmark_ticker})'])
print(metrics_df.to_string(index=False))

## Cumulative Growth Comparison

In [ ]:
cum_p = (1 + p_ret).cumprod()
cum_b = (1 + b_ret).cumprod()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(cum_p.index, cum_p.values, color='royalblue', linewidth=2.5, label='Portfolio')
ax.plot(cum_b.index, cum_b.values, color='orange',    linewidth=1.5, label=benchmark_ticker, alpha=0.8)
ax.set_ylabel('Cumulative Return (×)')
ax.set_title(f'Cumulative Growth: Portfolio vs {benchmark_ticker}')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}×'))
plt.tight_layout()
plt.show()

## 252-Day Rolling Beta

A rolling beta measures how sensitive the portfolio has been to benchmark movements over different market regimes.

In [ ]:
window = 252
rolling_betas = []
beta_dates    = []
for i in range(window, len(p_ret)):
    pw = p_ret.iloc[i-window:i].values
    bw = b_ret.iloc[i-window:i].values
    cov = np.cov(pw, bw)
    rolling_betas.append(cov[0,1] / cov[1,1] if cov[1,1] != 0 else float('nan'))
    beta_dates.append(p_ret.index[i])

if rolling_betas:
    rb = pd.Series(rolling_betas, index=beta_dates)
    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(rb.index, rb.values, color='orange', linewidth=1.5)
    ax.axhline(1, color='gray',  linewidth=1.0, linestyle='-', alpha=0.7, label='β = 1')
    ax.axhline(0, color='gray',  linewidth=0.5, alpha=0.4)
    ax.set_ylabel('Beta')
    ax.set_title(f'{window}-Day Rolling Beta vs {benchmark_ticker}')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f'Not enough data for {window}-day rolling beta.')

## Excess Returns Scatter

Each point represents one trading day. The regression line shows the relationship between portfolio and benchmark returns, with slope = beta and intercept = daily alpha.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(b_ret.values * 100, p_ret.values * 100,
           color='steelblue', alpha=0.4, s=5, linewidths=0)

xlim = max(abs(b_ret.values).max() * 100, 1)
x_line = np.linspace(-xlim, xlim, 200)
daily_alpha = a_val / 252
ax.plot(x_line, b_val * x_line + daily_alpha * 100,
        color='red', linewidth=1.5, alpha=0.6,
        label=f'Regression (β={b_val:.2f}, α={a_val:.2%}/yr)')

ax.axhline(0, color='gray', linewidth=0.8, alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, alpha=0.5)
ax.set_xlabel(f'{benchmark_ticker} Daily Return (%)')
ax.set_ylabel('Portfolio Daily Return (%)')
ax.set_title(f'Portfolio vs {benchmark_ticker} - Daily Returns')
ax.legend(fontsize=10)
ax.set_xlim(-xlim, xlim)
ax.set_ylim(-xlim, xlim)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Correlation: {corr_coef:.4f}  |  R²: {corr_coef**2:.4f}')